# Full Experiment Summary

This notebook explored multiple approaches for **binary Arabic poem era classification** using the `arbml/ashaar` dataset.

---

## Task Definition

### Input and target
- **Input text column**: `poem verses`
- **Target label column**: `poet era`

### Binary label mapping
To match the binary setting used earlier:
- **Modern** = `العصر الحديث`
- **Classical** = all other non-null eras

Rows with missing `poet era` were removed.

### Final labeled dataset
- Total usable labeled rows: **147,420**
- Class distribution:
  - **Classical**: 92,283
  - **Modern**: 55,137


# Experiments

---

## Experiment 1 ANN + TF-IDF

### What was used
- Text features: **TF-IDF**
- Max features: **15,000**
- Model:
  - `Dense(32, activation="relu")`
  - `Dropout(0.7)`
  - `Dense(1, activation="sigmoid")`
- Optimizer: `Adam`
- Loss: `binary_crossentropy`
- Epochs: `10`
- Batch size: `512`
- Early stopping: yes

### Why this experiment
This was the first strong neural baseline using sparse lexical features.

### Final full-test results
- **Accuracy**: `0.8107`
- **Macro Precision**: `0.8044`
- **Macro Recall**: `0.7841`
- **Macro F1**: `0.7916`

### Interpretation
- Strong baseline
- Good balance
- Best ANN experiment in terms of **Macro F1** among the simple sparse-feature ANN models

---

## Experiment 2 ANN + Bag-of-Words

### What was used
- Text features: **Bag-of-Words**
- Max features: **15,000**
- Model:
  - `Dense(32, activation="relu")`
  - `Dropout(0.7)`
  - `Dense(1, activation="sigmoid")`
- Optimizer: `Adam`
- Loss: `binary_crossentropy`
- Epochs: `10`
- Batch size: `512`
- Early stopping: yes

### Why this experiment
To compare raw word-count features against TF-IDF using the same ANN architecture.

### Final full-test results
- **Accuracy**: `0.8121`
- **Macro Precision**: `0.8110`
- **Macro Recall**: `0.7801`
- **Macro F1**: `0.7902`

### Interpretation
- Slightly better than TF-IDF in **Accuracy**
- Slightly lower than TF-IDF in **Macro F1**
- Very competitive baseline

---

## Experiment 3 FastText + ANN

### What was used
- Trained a **FastText** model on the cleaned training texts
- Converted each poem into a fixed-size vector by averaging word embeddings
- Prepared:
  - `X_train_ft`
  - `X_val_ft`
  - `X_test_ft`

### Why this experiment
To test dense semantic document vectors instead of sparse lexical vectors.

### Status
- Feature extraction was completed successfully
- ANN training/evaluation was **not finalized in the notebook comparison stage**

### Interpretation
- This experiment was exploratory
- It was not included in the final ranked comparison because no final full-test result was recorded

---

## Experiment 4 Linear SVM + TF-IDF

### What was used
- Text features: **TF-IDF**
- Classifier: `LinearSVC`
- Class weighting: `balanced`

### Why this experiment
To test a strong traditional machine learning baseline for text classification.

### Final full-test results
- **Accuracy**: `0.7865`
- **Macro Precision**: `0.7720`
- **Macro Recall**: `0.7781`
- **Macro F1**: `0.7746`

### Interpretation
- Simple and fast
- Performed worse than the best neural approaches
- Still useful as a classical baseline

---

## Experiment 5 BiGRU

### What was used
- Word-level tokenization
- Vocabulary size: **20,000**
- Original max sequence length was too large, so a practical cap was used:
  - chosen from training distribution
  - **MAX_LEN = 441** based on the 95th percentile
- Model:
  - `Embedding`
  - `Bidirectional(GRU(32, return_sequences=True))`
  - `Dropout(0.7)`
  - `Bidirectional(GRU(32))`
  - `Dropout(0.7)`
  - `Dense(1, activation="sigmoid")`
- Optimizer: `RMSprop`
- Loss: `binary_crossentropy`
- Epochs: `10`
- Batch size: `128`
- Early stopping: yes

### Why this experiment
To test a sequence model that captures contextual order directly from token sequences.

### Final full-test results
- **Accuracy**: `0.8142`
- **Macro Precision**: `0.8118`
- **Macro Recall**: `0.7839`
- **Macro F1**: `0.7934`

### Interpretation
- Best model before trying BERT
- Best non-transformer sequence model
- Slightly better than both ANN baselines

---

## Experiment 6 Arabic BERT (small subset)

### What was used
- Model: `aubmindlab/bert-base-arabertv02`
- Fine-tuned for binary classification
- Max length: `256`
- Training subset:
  - Train: `20,000`
  - Validation: `4,000`
  - Test: `4,000`

### Why this experiment
To test whether a transformer-based Arabic model could outperform the previous models before spending compute on the full dataset.

### Test-subset results
- **Accuracy**: `0.8113`
- **Macro Precision**: `0.8028`
- **Macro Recall**: `0.7876`
- **Macro F1**: `0.7936`

### Interpretation
- Promising result
- Very competitive
- Encouraged running BERT on the full dataset

---

## Experiment 7  Arabic BERT on the full dataset (final BERTerav2)

### What was used
- Model: `aubmindlab/bert-base-arabertv02`
- Fine-tuned for binary classification
- Full train/validation/test sets
- Max length: `256`
- Training setup:
  - learning rate: `2e-5`
  - batch size: `8`
  - epochs: `3`
  - weight decay: `0.01`
  - best model selected by **Macro F1**

### Why this experiment
To perform the final transformer experiment on the full dataset and compare fairly against the other full-test models.

### Final full-test results
- **Accuracy**: `0.8325`
- **Macro Precision**: `0.8208`
- **Macro Recall**: `0.8224`
- **Macro F1**: `0.8216`

### Classification report
- **Classical**
  - Precision: `0.87`
  - Recall: `0.86`
  - F1: `0.87`
- **Modern**
  - Precision: `0.77`
  - Recall: `0.78`
  - F1: `0.78`

### Confusion matrix
\[
\begin{bmatrix}
11936 & 1906 \\
1799 & 6472
\end{bmatrix}
\]

### Interpretation
- Best model overall
- Best accuracy
- Best macro F1
- Best balance between the two classes
- Clearly stronger than ANN and BiGRU on the full test set

---

# Final Comparison

| Experiment | Model | Features / Encoder | Test Size | Accuracy | Macro Precision | Macro Recall | Macro F1 |
|---|---|---|---:|---:|---:|---:|---:|
| 1 | ANN | TF-IDF | Full | 0.8107 | 0.8044 | 0.7841 | 0.7916 |
| 2 | ANN | Bag-of-Words | Full | 0.8121 | 0.8110 | 0.7801 | 0.7902 |
| 3 | Linear SVM | TF-IDF | Full | 0.7865 | 0.7720 | 0.7781 | 0.7746 |
| 4 | BiGRU | Word sequences | Full | 0.8142 | 0.8118 | 0.7839 | 0.7934 |
| 5 | BERT (subset) | AraBERT v0.2 | 4,000 | 0.8113 | 0.8028 | 0.7876 | 0.7936 |
| 6 | **BERTerav2** | **AraBERT v0.2** | **Full** | **0.8325** | **0.8208** | **0.8224** | **0.8216** |

---


# Final Ranking

1. **BERTerav2** — final best model  
2. **BiGRU** — best non-transformer model  
3. **ANN + TF-IDF** — best lightweight baseline  
4. **ANN + Bag-of-Words**  
5. **Linear SVM + TF-IDF**

---

# Final Conclusion

The experiments showed that:
- sparse lexical models were already strong
- sequence modeling with **BiGRU** improved performance further
- full fine-tuning of **Arabic BERT** produced the best final results by a clear margin

### Final selected model for deployment:
# **BERTerav2**

In [1]:
!pip -q install datasets scikit-learn tensorflow pandas numpy nltk

In [2]:
import re
import html
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
dataset = load_dataset("arbml/ashaar")
dataset

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type'],
        num_rows: 254630
    })
})

# Inspect dataset columns
This cell checks the dataset structure and prints the available columns.

In [5]:
train_df = dataset["train"].to_pandas()

print("Shape:", train_df.shape)
print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head(3))

Shape: (254630, 12)

Columns:
['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type']


,poem title,poem meter,poem verses,poem theme,poem url,poet name,poet description,poet url,poet era,poet location,poem description,poem language type
0,أصبح الملك للذي فطر الخلق,بحر الخفيف,"[أَصبَحَ المُلك لِلَّذي فَطر الخَل, قَ بِتَقدي...",قصيدة دينية,https://www.aldiwan.net/poem16182.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
1,من أي مولى ارتجي,بحر مجزوء الرمل,"[مِن أَي مَولى اِرتَجي, وَلاي باب التَجي, وَال...",قصيدة دينية,https://www.aldiwan.net/poem16183.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
2,العبد عبدك يا من أنت سيده,بحر البسيط,"[العَبد عَبدك يا مَن أَنتَ سَيدهُ, وَلَيسَ غَي...",قصيدة ذم,https://www.aldiwan.net/poem16184.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None


# Inspect era labels
This cell shows all unique values in the era column and their frequencies, so we can build the binary mapping correctly.

In [6]:
era_counts = train_df["poet era"].value_counts(dropna=False)

print("Number of unique era labels:", train_df["poet era"].nunique(dropna=True))
print("\nUnique era labels:")
print(train_df["poet era"].dropna().unique())

print("\nEra counts:")
display(era_counts)

Number of unique era labels: 14

Unique era labels:
['العصر العثماني' 'العصر الجاهلي' 'العصر العباسي' 'العصر الحديث'
 'العصر الأندلسي' 'العصر المملوكي' 'قبل الإسلام' 'العصر الأموي'
 'العصر الفاطمي' 'العصر الإسلامي' 'العصر الأيوبي' 'المغرب والأندلس'
 'المخضرمين' 'عصر بين الدولتين']

Era counts:


,count
poet era,
None,107209
العصر الحديث,55137
العصر العباسي,30413
العصر المملوكي,19059
العصر العثماني,11872
المغرب والأندلس,6614
العصر الفاطمي,4939
العصر الأندلسي,4770
العصر الأموي,3958


# Build binary era labels
This cell removes rows with missing era labels, then maps the dataset into two classes:
`Modern` and `Classical`.

In [7]:
df = train_df.copy()

# Remove rows with missing era labels
df = df[df["poet era"].notna()].copy()

# Binary mapping: modern vs classical
df["era_binary"] = df["poet era"].apply(
    lambda x: "Modern" if x == "العصر الحديث" else "Classical"
)

print("Shape after removing missing labels:", df.shape)
print("\nBinary label counts:")
print(df["era_binary"].value_counts())

display(df[["poem verses", "poet era", "era_binary"]].head(5))

Shape after removing missing labels: (147421, 13)

Binary label counts:
era_binary
Classical    92284
Modern       55137
Name: count, dtype: int64


,poem verses,poet era,era_binary
0,"[أَصبَحَ المُلك لِلَّذي فَطر الخَل, قَ بِتَقدي...",العصر العثماني,Classical
1,"[مِن أَي مَولى اِرتَجي, وَلاي باب التَجي, وَال...",العصر العثماني,Classical
2,"[العَبد عَبدك يا مَن أَنتَ سَيدهُ, وَلَيسَ غَي...",العصر العثماني,Classical
3,"[لَو كُنتَ أَطمَع بِالمَنام تَوهما, لَسالَت طَ...",العصر العثماني,Classical
4,"[يعد عَليَّ أَنفاسي ذُنوباً, إِذا ما قُلت أَفد...",العصر العثماني,Classical


# Convert verses to plain text
This cell converts the `poem verses` field into a clean plain-text string so it can be used as model input.

In [9]:
import ast
import numpy as np

def verses_to_text(x):
    # Handle None
    if x is None:
        return ""

    # Handle NaN scalars safely
    if isinstance(x, float) and pd.isna(x):
        return ""

    # If already list/array-like
    if isinstance(x, (list, tuple, np.ndarray)):
        return " ".join([str(v).strip() for v in x if str(v).strip()])

    # If string representation of a list
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return ""
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, (list, tuple, np.ndarray)):
                return " ".join([str(v).strip() for v in parsed if str(v).strip()])
        except:
            pass
        return x

    return str(x)

df["text_raw"] = df["poem verses"].apply(verses_to_text)

print("Sample converted texts:\n")
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print(df["text_raw"].iloc[i][:500])
    print()

Sample converted texts:

--- Sample 1 ---
أَصبَحَ المُلك لِلَّذي فَطر الخَل قَ بِتَقديرٍ للعَزيز العَليمِ غافر الذَنب للمسيءِ بِعَفوٍ قابل التَوب ذي العَطاء العَميمِ مُرسل المُصطَفى البَشير إِلَينا رَحمة مِنهُ بِالكَلام القَديمِ رَبَنا رَبّنا إِلَيكَ أَنينا فَأَجرنا مِن حَر نار الجَحيمِ وَاكفِنا شَرّ ما نَخاف بِلُطفٍ يا عَظيماً يَرجى لِكُل عَظيمِ وَتَقبل أَعمالَنا وَاعفُ عَنا وَأَنلنا دُخول دار النَعيمِ بِنَبي بَعثَتهُ فَهَدانا لِصِراط مِن الهُدى مُستَقيمِ وَبِمَن نَحنُ في حِماهُ مَدى الدَهر أَخيهِ يَحيى الحصور الكَريمِ أَدرك أَدرك قَوم

--- Sample 2 ---
مِن أَي مَولى اِرتَجي وَلاي باب التَجي وَاللَهُ حَيٌّ رازِقٌ يُعطي الجَزيل لمُرتَجي رَب جَواد لَم يَزَل مِن كُل ضيقٍ مَخرَجي إِن رُحت أَرجوغَيرَهُ خابَ الرواح مَع المَجي يا عَيس آمالي أَقصدي باب الكَريم وَعَرجي وَضَعي رِحالك وَاِرتَعي فَالأُم حَمل المُزعجِ وَتَوسَلي بِمُحمدٍ وَبِآلهِ كَي تَنتجي الهاشمي المُصطَفى صج الهُدى المُتَبَلِجِ وَبِشَيبة الصَديق صا حب كل فَضل أَبهَجِ وَالسَيد الفاروق مِن بِسِوى الهُدى لَم يَلهجِ وَبصنوه عُثمان ذ

# Apply Arabic text preprocessing
This cell applies the same core preprocessing used in the paper:
remove HTML, diacritics, kashida, punctuation, symbols, and digits.
Stop words are kept.

In [10]:
arabic_diacritics = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')
kashida_char = '\u0640'

def clean_arabic_text(text):
    if text is None:
        return ""

    text = html.unescape(str(text))

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove diacritics
    text = re.sub(arabic_diacritics, "", text)

    # Remove kashida
    text = text.replace(kashida_char, "")

    # Normalize whitespace early
    text = re.sub(r"\s+", " ", text).strip()

    # Remove digits (Arabic and English)
    text = re.sub(r"[0-9٠-٩]", " ", text)

    # Keep Arabic letters and spaces only
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["text_clean"] = df["text_raw"].apply(clean_arabic_text)

print("Sample cleaned texts:\n")
for i in range(3):
    print(f"--- Sample {i+1} ---")
    print(df["text_clean"].iloc[i][:500])
    print()

Sample cleaned texts:

--- Sample 1 ---
أصبح الملك للذي فطر الخل ق بتقدير للعزيز العليم غافر الذنب للمسيء بعفو قابل التوب ذي العطاء العميم مرسل المصطفى البشير إلينا رحمة منه بالكلام القديم ربنا ربنا إليك أنينا فأجرنا من حر نار الجحيم واكفنا شر ما نخاف بلطف يا عظيما يرجى لكل عظيم وتقبل أعمالنا واعف عنا وأنلنا دخول دار النعيم بنبي بعثته فهدانا لصراط من الهدى مستقيم وبمن نحن في حماه مدى الدهر أخيه يحيى الحصور الكريم أدرك أدرك قوما أتوا بافتقار وانكسار ومدمع مسجوم شهدت أرواحهم أنك الله وجاءوا بكل قلب سليم

--- Sample 2 ---
من أي مولى ارتجي ولاي باب التجي والله حي رازق يعطي الجزيل لمرتجي رب جواد لم يزل من كل ضيق مخرجي إن رحت أرجوغيره خاب الرواح مع المجي يا عيس آمالي أقصدي باب الكريم وعرجي وضعي رحالك وارتعي فالأم حمل المزعج وتوسلي بمحمد وبآله كي تنتجي الهاشمي المصطفى صج الهدى المتبلج وبشيبة الصديق صا حب كل فضل أبهج والسيد الفاروق من بسوى الهدى لم يلهج وبصنوه عثمان ذي الن نورين أقوم منهج وعلي الكرار فا تح كل باب مرتج وبقية الصحب الكرا م أولي الثنا المتأرج هم أبحر الفضل الذي ن بغيرهم لم تفرج و

# Prepare features and labels
This cell removes empty texts, keeps the cleaned text as input, and encodes the binary labels.

In [11]:
df = df[df["text_clean"].str.strip() != ""].copy()

label_map = {"Classical": 0, "Modern": 1}
df["label"] = df["era_binary"].map(label_map)

X = df["text_clean"].values
y = df["label"].values

print("Final dataset shape:", df.shape)
print("\nLabel distribution:")
print(df["era_binary"].value_counts())

print("\nEncoded label distribution:")
print(pd.Series(y).value_counts().sort_index())

display(df[["text_clean", "era_binary", "label"]].head(3))

Final dataset shape: (147420, 16)

Label distribution:
era_binary
Classical    92283
Modern       55137
Name: count, dtype: int64

Encoded label distribution:
0    92283
1    55137
Name: count, dtype: int64


,text_clean,era_binary,label
0,أصبح الملك للذي فطر الخل ق بتقدير للعزيز العلي...,Classical,0
1,من أي مولى ارتجي ولاي باب التجي والله حي رازق ...,Classical,0
2,العبد عبدك يا من أنت سيده وليس غيرك في الأوصاب...,Classical,0


# Split into train, validation, and test
This cell creates the train/validation/test split following the paper:
15% test, and 15% of the remaining training portion for validation.

In [12]:
from sklearn.model_selection import train_test_split

# Step 1: test split = 15%
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=SEED,
    stratify=y
)

# Step 2: validation split = 15% of the remaining 85%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_val
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))
print("Test size:", len(X_test))

print("\nTrain label distribution:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nValidation label distribution:")
print(pd.Series(y_val).value_counts().sort_index())

print("\nTest label distribution:")
print(pd.Series(y_test).value_counts().sort_index())

Train size: 106510
Validation size: 18797
Test size: 22113

Train label distribution:
0    66674
1    39836
Name: count, dtype: int64

Validation label distribution:
0    11767
1     7030
Name: count, dtype: int64

Test label distribution:
0    13842
1     8271
Name: count, dtype: int64


# Vectorize text using TF-IDF
This cell converts the cleaned text into TF-IDF vectors with a maximum vocabulary size of 15,000, following the paper.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=15000
)

X_train_vec = vectorizer.fit_transform(X_train)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print("X_train_vec shape:", X_train_vec.shape)
print("X_val_vec shape:", X_val_vec.shape)
print("X_test_vec shape:", X_test_vec.shape)

print("\nVocabulary size learned:", len(vectorizer.vocabulary_))

X_train_vec shape: (106510, 15000)
X_val_vec shape: (18797, 15000)
X_test_vec shape: (22113, 15000)

Vocabulary size learned: 15000


# Build the ANN model
This cell builds the feed-forward ANN used for binary classification.

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

input_dim = X_train_vec.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dropout(0.7),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │       480,032 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 480,065 (1.83 MB)

 Trainable params: 480,065 (1.83 MB)

 Non-trainable params: 0 (0.00 B)

# Train the ANN model
This cell trains the ANN for up to 10 epochs, uses early stopping, and saves the best model based on validation accuracy.

In [15]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_ann_binary.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train_vec.toarray(),
    y_train,
    validation_data=(X_val_vec.toarray(), y_val),
    epochs=10,
    batch_size=512,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

Epoch 1/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6333 - loss: 0.6508
Epoch 1: val_accuracy improved from None to 0.76714, saving model to best_ann_binary.keras

Epoch 1: finished saving model to best_ann_binary.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 8s 28ms/step - accuracy: 0.6746 - loss: 0.6111 - val_accuracy: 0.7671 - val_loss: 0.5221
Epoch 2/10
208/209 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7686 - loss: 0.5099
Epoch 2: val_accuracy improved from 0.76714 to 0.80428, saving model to best_ann_binary.keras

Epoch 2: finished saving model to best_ann_binary.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.7819 - loss: 0.4895 - val_accuracy: 0.8043 - val_loss: 0.4469
Epoch 3/10
208/209 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8071 - loss: 0.4441
Epoch 3: val_accuracy improved from 0.80428 to 0.81002, saving model to best_ann_binary.keras

Epoch 3: finished saving model to best_ann_binary.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 

# Evaluate on the test set
This cell loads the best model, predicts on the test set, and reports the final binary classification metrics.

In [16]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

best_model = load_model("best_ann_binary.keras")

y_proba = best_model.predict(X_test_vec.toarray(), batch_size=512, verbose=1)
y_pred = (y_proba >= 0.5).astype("int32").reshape(-1)

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro"
)

print(f"Test Accuracy:  {accuracy:.4f}")
print(f"Macro Precision:{precision:.4f}")
print(f"Macro Recall:   {recall:.4f}")
print(f"Macro F1:       {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step
Test Accuracy:  0.8107
Macro Precision:0.8044
Macro Recall:   0.7841
Macro F1:       0.7916

Classification Report:
              precision    recall  f1-score   support

   Classical       0.82      0.89      0.85     13842
      Modern       0.79      0.68      0.73      8271

    accuracy                           0.81     22113
   macro avg       0.80      0.78      0.79     22113
weighted avg       0.81      0.81      0.81     22113

Confusion Matrix:
[[12318  1524]
 [ 2661  5610]]


In [17]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Target folder and file path
save_dir = "/content/drive/MyDrive/lsen"
os.makedirs(save_dir, exist_ok=True)

source_model_path = "best_ann_binary.keras"
target_model_path = os.path.join(save_dir, "era-ann.keras")

# Copy the best saved model
shutil.copy2(source_model_path, target_model_path)

print("Model saved to:")
print(target_model_path)

Mounted at /content/drive
Model saved to:
/content/drive/MyDrive/lsen/era-ann.keras


In [64]:
def predict_era(poem_text, model, vectorizer):
    cleaned_text = clean_arabic_text(poem_text)
    poem_vec = vectorizer.transform([cleaned_text]).toarray()

    prob = model.predict(poem_vec, verbose=0)[0][0]
    pred_label = "Modern" if prob >= 0.5 else "Classical"

    return {
        "original_text": poem_text,
        "cleaned_text": cleaned_text,
        "modern_probability": float(prob),
        "classical_probability": float(1 - prob),
        "predicted_label": pred_label
    }

# Put your poem here
sample_poem = """
لآخرِ أمر المـرء تُفضي أَوائِلُـه
و دهْرُكَ في الإدبار كُثر وسائلُهْ
جَرَعتُ كؤوس الهمِّ حتى ألِفْتُها
و قد ضاق ذرعا بي وأُثْقِلَ كاهِلُهْ
وأَبْدى ليَ الأصحابُ ما لسْتُ جاهلاً
بأنَّ اشْتِدادَ الكرب فَرداً تُعَامِلُـهْ
وأنْ لا وفاء غير في الأُنس والرَّخا
وإلّا فَكُلٌّ قَد توالَت مَشاغِلُـه
عَرَفتُ من الأيام تكرار غدرِها
فمَن قالَ أنَّ الغدر قد خابَ فاعِلُهْ
فما خابت الأيامُ من سوء فعلها
وقدخاب حُسْنُ الظَّن خابَ تفاؤلُه
فيا ظَهري المطعون بالخنجر الذّي
على خيري الموصول قد شَبَّ حامِلُه
أيَا ظَهري المطعون خيراً فعَلْت بي
بألّا تُريني الغدر وجهاً أُقابِلُهْ
أيا فالقَ الإصباحِ والحَبِّ والنَّوى
ويا سابقاً للعفو تُخْشَى نَوازِلُهْ
فَلَم يَبقَ إلّا أنتَ منجا ومُرْتجى
وإلّا فَكُلٌّ قد توَلَّت قوافِلُه
"""

result = predict_era(sample_poem, best_model, vectorizer)

print("Predicted label:", result["predicted_label"])
print("Modern probability:", round(result["modern_probability"], 4))
print("Classical probability:", round(result["classical_probability"], 4))
print("\nCleaned text:")
print(result["cleaned_text"][:1000])

Predicted label: Modern
Modern probability: 0.707
Classical probability: 0.293

Cleaned text:
لآخر أمر المرء تفضي أوائله و دهرك في الإدبار كثر وسائله جرعت كؤوس الهم حتى ألفتها و قد ضاق ذرعا بي وأثقل كاهله وأبدى لي الأصحاب ما لست جاهلا بأن اشتداد الكرب فردا تعامله وأن لا وفاء غير في الأنس والرخا وإلا فكل قد توالت مشاغله عرفت من الأيام تكرار غدرها فمن قال أن الغدر قد خاب فاعله فما خابت الأيام من سوء فعلها وقدخاب حسن الظن خاب تفاؤله فيا ظهري المطعون بالخنجر الذي على خيري الموصول قد شب حامله أيا ظهري المطعون خيرا فعلت بي بألا تريني الغدر وجها أقابله أيا فالق الإصباح والحب والنوى ويا سابقا للعفو تخشى نوازله فلم يبق إلا أنت منجا ومرتجى وإلا فكل قد تولت قوافله


# Vectorize the cleaned text using FastText
This cell trains a FastText model on the cleaned training texts and converts each poem into a fixed-size document vector by averaging word embeddings.

In [35]:
!pip -q install gensim


In [36]:
from gensim.models import FastText
import numpy as np

# Tokenize texts
train_tokens = [text.split() for text in X_train]
val_tokens   = [text.split() for text in X_val]
test_tokens  = [text.split() for text in X_test]

# Train FastText on TRAIN only to avoid leakage
fasttext_model = FastText(
    sentences=train_tokens,
    vector_size=300,
    window=5,
    min_count=2,
    workers=4,
    sg=1,          # skip-gram
    epochs=10,
    seed=SEED
)

def document_vector(tokens, model, vector_size=300):
    vectors = [model.wv[token] for token in tokens if token in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32)

# Convert each text into one fixed-size vector
X_train_ft = np.vstack([document_vector(tokens, fasttext_model, 300) for tokens in train_tokens])
X_val_ft   = np.vstack([document_vector(tokens, fasttext_model, 300) for tokens in val_tokens])
X_test_ft  = np.vstack([document_vector(tokens, fasttext_model, 300) for tokens in test_tokens])

print("FastText vector shapes:")
print("X_train_ft:", X_train_ft.shape)
print("X_val_ft:  ", X_val_ft.shape)
print("X_test_ft: ", X_test_ft.shape)

print("\nExample vector (first 10 values):")
print(X_train_ft[0][:10])

FastText vector shapes:
X_train_ft: (106510, 300)
X_val_ft:   (18797, 300)
X_test_ft:  (22113, 300)

Example vector (first 10 values):
[-0.40151972  0.02384541 -0.15524131 -0.26032162 -0.01108392  0.02671668
  0.01849084  0.27309138  0.0470147   0.01952297]


# Train ANN using FastText vectors
This cell trains the same ANN architecture using FastText document vectors instead of TF-IDF features.

In [38]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

fasttext_ann2 = Sequential([
    Input(shape=(X_train_ft.shape[1],)),
    Dense(32, activation="relu"),
    Dropout(0.7),
    Dense(1, activation="sigmoid")
])

fasttext_ann2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ft = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

checkpoint_ft = ModelCheckpoint(
    "best_ann_fasttext.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history_ft2 = fasttext_ann2.fit(
    X_train_ft,
    y_train,
    validation_data=(X_val_ft, y_val),
    epochs=20,
    batch_size=512,
    callbacks=[early_stopping_ft, checkpoint_ft],
    verbose=1
)

Epoch 1/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6337 - loss: 0.6328
Epoch 1: val_accuracy improved from None to 0.74847, saving model to best_ann_fasttext.keras

Epoch 1: finished saving model to best_ann_fasttext.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.6633 - loss: 0.6006 - val_accuracy: 0.7485 - val_loss: 0.5373
Epoch 2/20
191/209 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7284 - loss: 0.5446
Epoch 2: val_accuracy improved from 0.74847 to 0.77539, saving model to best_ann_fasttext.keras

Epoch 2: finished saving model to best_ann_fasttext.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7408 - loss: 0.5359 - val_accuracy: 0.7754 - val_loss: 0.4975
Epoch 3/20
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7568 - loss: 0.5205
Epoch 3: val_accuracy improved from 0.77539 to 0.78087, saving model to best_ann_fasttext.keras

Epoch 3: finished saving model to best_ann_fasttext.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - ac

# Vectorize text using Bag-of-Words
This cell converts the cleaned text into Bag-of-Words vectors with a maximum vocabulary size of 15,000.

In [39]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(
    max_features=15000
)

X_train_bow = bow_vectorizer.fit_transform(X_train)
X_val_bow = bow_vectorizer.transform(X_val)
X_test_bow = bow_vectorizer.transform(X_test)

print("X_train_bow shape:", X_train_bow.shape)
print("X_val_bow shape:", X_val_bow.shape)
print("X_test_bow shape:", X_test_bow.shape)

print("\nVocabulary size learned:", len(bow_vectorizer.vocabulary_))

X_train_bow shape: (106510, 15000)
X_val_bow shape: (18797, 15000)
X_test_bow shape: (22113, 15000)

Vocabulary size learned: 15000


# Train ANN using Bag-of-Words
This cell trains the same ANN architecture using Bag-of-Words vectors instead of TF-IDF features.

In [40]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

bow_ann = Sequential([
    Input(shape=(X_train_bow.shape[1],)),
    Dense(32, activation="relu"),
    Dropout(0.7),
    Dense(1, activation="sigmoid")
])

bow_ann.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping_bow = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

checkpoint_bow = ModelCheckpoint(
    "best_ann_bow.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history_bow = bow_ann.fit(
    X_train_bow.toarray(),
    y_train,
    validation_data=(X_val_bow.toarray(), y_val),
    epochs=10,
    batch_size=512,
    callbacks=[early_stopping_bow, checkpoint_bow],
    verbose=1
)

Epoch 1/10
209/209 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6834 - loss: 0.6186
Epoch 1: val_accuracy improved from None to 0.79827, saving model to best_ann_bow.keras

Epoch 1: finished saving model to best_ann_bow.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - accuracy: 0.7368 - loss: 0.5637 - val_accuracy: 0.7983 - val_loss: 0.4597
Epoch 2/10
208/209 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7984 - loss: 0.4602
Epoch 2: val_accuracy improved from 0.79827 to 0.81508, saving model to best_ann_bow.keras

Epoch 2: finished saving model to best_ann_bow.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.8038 - loss: 0.4474 - val_accuracy: 0.8151 - val_loss: 0.4192
Epoch 3/10
207/209 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8219 - loss: 0.4093
Epoch 3: val_accuracy improved from 0.81508 to 0.81619, saving model to best_ann_bow.keras

Epoch 3: finished saving model to best_ann_bow.keras
209/209 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8240 - loss: 0

# Evaluate Bag-of-Words model on the test set
This cell loads the best BoW-based ANN model and reports the final test metrics.

In [41]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

best_bow_model = load_model("best_ann_bow.keras")

y_proba_bow = best_bow_model.predict(X_test_bow.toarray(), batch_size=512, verbose=1)
y_pred_bow = (y_proba_bow >= 0.5).astype("int32").reshape(-1)

accuracy_bow = accuracy_score(y_test, y_pred_bow)
precision_bow, recall_bow, f1_bow, _ = precision_recall_fscore_support(
    y_test, y_pred_bow, average="macro"
)

print(f"Test Accuracy:   {accuracy_bow:.4f}")
print(f"Macro Precision: {precision_bow:.4f}")
print(f"Macro Recall:    {recall_bow:.4f}")
print(f"Macro F1:        {f1_bow:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_bow,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_bow))

44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step
Test Accuracy:   0.8121
Macro Precision: 0.8110
Macro Recall:    0.7801
Macro F1:        0.7902

Classification Report:
              precision    recall  f1-score   support

   Classical       0.81      0.91      0.86     13842
      Modern       0.81      0.65      0.72      8271

    accuracy                           0.81     22113
   macro avg       0.81      0.78      0.79     22113
weighted avg       0.81      0.81      0.81     22113

Confusion Matrix:
[[12558  1284]
 [ 2870  5401]]


In [42]:
from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Target folder and file path
save_dir = "/content/drive/MyDrive/lsen"
os.makedirs(save_dir, exist_ok=True)

source_model_path = "best_ann_bow.keras"
target_model_path = os.path.join(save_dir, "annBOW.keras")

# Copy the saved best model
shutil.copy2(source_model_path, target_model_path)

print("Model saved to:")
print(target_model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to:
/content/drive/MyDrive/lsen/annBOW.keras


In [43]:
from tensorflow.keras.models import load_model

# Load the saved BoW model if needed
bow_model = load_model("best_ann_bow.keras")

def predict_era_bow(poem_text, model, vectorizer):
    cleaned_text = clean_arabic_text(poem_text)
    poem_vec = vectorizer.transform([cleaned_text]).toarray()

    prob = model.predict(poem_vec, verbose=0)[0][0]
    pred_label = "Modern" if prob >= 0.5 else "Classical"

    return {
        "original_text": poem_text,
        "cleaned_text": cleaned_text,
        "modern_probability": float(prob),
        "classical_probability": float(1 - prob),
        "predicted_label": pred_label
    }

# Put your poem here
sample_poem = """
هَلَّلَ وَجهُ الكَونِ وَاِبتَسَم السَعدُ
وَعادَ شَبابُ الدَهرِ وَاِنتَظَم العِقدُ
وَأَصبَحَتِ العَلياءُ يَفتَرُّ ثَغرُها
وَقد كانَ فيها عَن جَميع الوَرى صَدُّ
لَوَت جيدَها نَحوَ الذي كانَ كَفؤَها
سعودُ بني الدنيا الذي فِعلُه جِدُّ
رَأى فيهِ سُلطانُ المُلوكِ وَفَخرُها
مَخايِل مجدٍ حينَ ما ضمَّهُ المَهدُ
فَما زالَ يَنمو وَالفَضائِلُ تَرتَقي
إِلى أَن بدا في فَضلِهِ العلمُ الفردُ
نَجيبُ مَناجيبٍ وَفرعُ أَئِمَّةٍ
هُمُ القَومُ لا عُزلُ اليَدَينِ وَلا نُكدُ
حَبيبٌ إِلَيهِ الحِلمُ وَالجودُ وَالتُقى
بَغيضٌ إِلَيهِ الجَورُ وَالبُخلُ وَالحِقدُ
فَلَّما سَمَت فيهِ النَجابَةُ وَاِرتَقى
إِلى غايَةٍ ما فَوقَها لِلفَتى قَصدُ
وَحلَّ بِعَرشِ المَجدِ في شَرخِ عُمرِهِ
كَأَفعالِ آباءٍ لهُ وَهُمُ مُردُ
رَآهُ إِمامُ المُسلِمينَ لِعَهدِهِ
كَفِيّاً وَفيما قَد رَأى الحَزمُ وَالرُشدُ
فَوَلّاهُ عَهدَ المُسلِمينَ رِعايَةً
لِنُصحِهِمُ فيما يَغيبُ وَما يَبدو
فرَضيَ بَنو الإِسلامِ ذاكَ وَبايَعوا
وَقالوا عَلَينا الشُكرُ لِلَّهِ وَالحَمدُ
فَقامَ بِأَعباءِ الخِلافَةِ ماجِدٌ
كَما فَعَلَت آباؤُهُ قَبلُ والجَدُّ
مُلوكٌ سما ذا نحوَ ذا فَتَوافَقوا
عَلى أَنَّ ذا كَفٌّ وَهذا لهُ عَضدُ
فَلِلَّهِ يا عَبدَ العَزيزِ بن فَيصَلٍ
مَآثِرُ تَبقى ما بَقي في الوَرى عَبدُ
وَهِمَّةُ مِقدامٍ إِذا هَمَّ لم يَكُن
يُنَهنِهُهُ عَنها وَعيدٌ وَلا وَعدُ
نَصَرتُم بِها الإِسلامَ في كُلِّ مَوطِنٍ
وَسُدتُم بِها أَهلَ القُرى وَالذي يَبدو
مَلَكتُم بها ما بَينَ بُصرى وَأَبيَنٍ
وَمَدَّت لكُم أَعناقَها مِصرُ وَالهِندُ
فَلَم تَقبَلوا إِلّا مَواكِرَ مَجدِكُم
وَفي العَربِ العَربا لِمَن سادَها مَجدُ
إِذا رُمتُمُ أَمراً مَلَكتُم زِمامَهُ
وَإن تَقدَحوا لَم يَكبُ يَوماً لكُم زَندُ
فَكَيفَ وَأَنتُم عِصمَةُ الدينِ وَالدُنى
وَسادَتهُم مِن قَبلِ هذا وَمِن بَعدُ
أَقَمتُم قَناةَ الدينِ بِالسُمرِ وَالظُبى
وَشوسٍ بِهِم تَعدو مُطَهَّمةٌ جُردُ
سِراعٍ إِلى الهَيجا ثِقالٍ إِذا الوَغى
تَكَعكَعَ عن حَوماتِها الأَسدُ الوَردُ
إِذا جاهِلٌ أَغراهُ مِن سوءِ حَظِّهِ
بِأَن سَوف يُنجيهِ معَ الهَرَبِ البُعدُ
رَمَوهُ بِشَهبَها يُعجِزُ الطَيرَ سَيرُها
فَلَم يُنجِهِ غَورٌ وَلا جَبَلٌ صَلدُ
فَأَصبَحَ يَدو بِالثُبورِ وَيَمتَني
لَو اِن صارَ كَالعَنقاءِ أَو ضَمَّهُ لَحدُ
هُمُ ما همُ لا الذَخلُ يُدرِكُ عِندهُم
وَإِن طَلَبوهُ أَدرَكوهُ وَلا بُدُّ
وَكَم غُمَّةٍ قَد فَرَّجوها بِهِمَّةٍ
بِها قَبلَ مَسعاهُم عُيونُ الهدى رُمدُ
أَجاروا عَلى كِسرى بن ساسانَ ماضِياً
وَفي الغابِرينَ الآنَ لَيسَ لهُم نِدُّ
همُ بَهجَةُ الدُنيا وَكَوكَبُ سَعدِها
وَهُم خَيرُ مَن أُلقي له الحلُّ وَالعقدُ
إِذا وَهَبوا أَغنَوا وَإَن قَدِروا عَفَوا
وَإِن حارَبوا أَشجَوا وَإِن عَقدوا شَدّوا
عَطاءٌ وَلا مَنٌّ وَحُكمٌ وَلا هَوىً
وَفصلٌ وَلا هَزلٌ وَحِلمٌ وَلا حَردُ
فَللّهِ رَبّي الحَمدُ وَالشُكرُ وَالثَنا
عَلى نِعَمٍ لا يُستَطاعُ لها عَدُّ
وَعُذراً فَما مَدحي بِقاضٍ حُقوقَكُم
عَليَّ وَلا المِعشارَ لكِنَّهُ الجَهدُ
وَلا تَعدَمِ الدُنيا بَقاكُم عَلى المَدى
وَلا زالَ مِن إِحسانِكُم لِلوَرى رِفدُ
وَصَلِّ إلهَ العالَمينَ على الذي
لهُ الفَخرُ في الدُنيا وَفي الجَنَّةِ الخُلدُ
كَذا الآلُ وَالأَصحابُ ما قالَ مُنشِدٌ
تَهَلَّلَ وَجهُ الكَونِ وَاِبتَسَم السَعدُ

"""

result = predict_era_bow(sample_poem, bow_model, bow_vectorizer)

print("Predicted label:", result["predicted_label"])
print("Modern probability:", round(result["modern_probability"], 4))
print("Classical probability:", round(result["classical_probability"], 4))
print("\nCleaned text:")
print(result["cleaned_text"][:1000])

Predicted label: Modern
Modern probability: 0.9542
Classical probability: 0.0458

Cleaned text:
هلل وجه الكون وابتسم السعد وعاد شباب الدهر وانتظم العقد وأصبحت العلياء يفتر ثغرها وقد كان فيها عن جميع الورى صد لوت جيدها نحو الذي كان كفؤها سعود بني الدنيا الذي فعله جد رأى فيه سلطان الملوك وفخرها مخايل مجد حين ما ضمه المهد فما زال ينمو والفضائل ترتقي إلى أن بدا في فضله العلم الفرد نجيب مناجيب وفرع أئمة هم القوم لا عزل اليدين ولا نكد حبيب إليه الحلم والجود والتقى بغيض إليه الجور والبخل والحقد فلما سمت فيه النجابة وارتقى إلى غاية ما فوقها للفتى قصد وحل بعرش المجد في شرخ عمره كأفعال آباء له وهم مرد رآه إمام المسلمين لعهده كفيا وفيما قد رأى الحزم والرشد فولاه عهد المسلمين رعاية لنصحهم فيما يغيب وما يبدو فرضي بنو الإسلام ذاك وبايعوا وقالوا علينا الشكر لله والحمد فقام بأعباء الخلافة ماجد كما فعلت آباؤه قبل والجد ملوك سما ذا نحو ذا فتوافقوا على أن ذا كف وهذا له عضد فلله يا عبد العزيز بن فيصل مآثر تبقى ما بقي في الورى عبد وهمة مقدام إذا هم لم يكن ينهنهه عنها وعيد ولا وعد نصرتم بها الإسلام في كل مو

# Train and evaluate Linear SVM on TF-IDF
This cell trains a Linear SVM using the existing TF-IDF features and reports the test metrics.

In [44]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

svm_clf = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=SEED
)

svm_clf.fit(X_train_vec, y_train)

y_pred_svm = svm_clf.predict(X_test_vec)

accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm, recall_svm, f1_svm, _ = precision_recall_fscore_support(
    y_test, y_pred_svm, average="macro"
)

print(f"Test Accuracy:   {accuracy_svm:.4f}")
print(f"Macro Precision: {precision_svm:.4f}")
print(f"Macro Recall:    {recall_svm:.4f}")
print(f"Macro F1:        {f1_svm:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_svm,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_svm))

Test Accuracy:   0.7865
Macro Precision: 0.7720
Macro Recall:    0.7781
Macro F1:        0.7746

Classification Report:
              precision    recall  f1-score   support

   Classical       0.84      0.81      0.83     13842
      Modern       0.70      0.74      0.72      8271

    accuracy                           0.79     22113
   macro avg       0.77      0.78      0.77     22113
weighted avg       0.79      0.79      0.79     22113

Confusion Matrix:
[[11230  2612]
 [ 2110  6161]]


# Tokenize text for BiGRU
This cell converts the cleaned text into integer word sequences, limits the vocabulary to 20,000 words, and pads all sequences to the same length.

In [45]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

VOCAB_SIZE = 20000

tokenizer_rnn = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer_rnn.fit_on_texts(X_train)

train_seq = tokenizer_rnn.texts_to_sequences(X_train)
val_seq = tokenizer_rnn.texts_to_sequences(X_val)
test_seq = tokenizer_rnn.texts_to_sequences(X_test)

# Use the training set to define max length
max_len = max(len(seq) for seq in train_seq)

X_train_seq = pad_sequences(train_seq, maxlen=max_len, padding="post", truncating="post")
X_val_seq = pad_sequences(val_seq, maxlen=max_len, padding="post", truncating="post")
X_test_seq = pad_sequences(test_seq, maxlen=max_len, padding="post", truncating="post")

print("Vocabulary size used:", min(VOCAB_SIZE, len(tokenizer_rnn.word_index) + 1))
print("Max sequence length:", max_len)

print("\nSequence shapes:")
print("X_train_seq:", X_train_seq.shape)
print("X_val_seq:  ", X_val_seq.shape)
print("X_test_seq: ", X_test_seq.shape)

print("\nExample tokenized sequence (first 30 tokens):")
print(X_train_seq[0][:30])

Vocabulary size used: 20000
Max sequence length: 55179

Sequence shapes:
X_train_seq: (106510, 55179)
X_val_seq:   (18797, 55179)
X_test_seq:  (22113, 55179)

Example tokenized sequence (first 30 tokens):
[    1  2752     1     1   764 19842  1314 16145     1     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0]


# Re-pad sequences with a practical max length
This cell replaces the raw maximum length with a practical capped length based on the training length distribution.

In [46]:
train_lengths = np.array([len(seq) for seq in train_seq])

print("Train sequence length stats:")
print("Min:", train_lengths.min())
print("Median:", int(np.median(train_lengths)))
print("95th percentile:", int(np.percentile(train_lengths, 95)))
print("99th percentile:", int(np.percentile(train_lengths, 99)))
print("Max:", train_lengths.max())

MAX_LEN = int(np.percentile(train_lengths, 95))

X_train_seq = pad_sequences(train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_seq = pad_sequences(val_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_seq = pad_sequences(test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("\nChosen MAX_LEN:", MAX_LEN)
print("X_train_seq:", X_train_seq.shape)
print("X_val_seq:  ", X_val_seq.shape)
print("X_test_seq: ", X_test_seq.shape)

Train sequence length stats:
Min: 3
Median: 47
95th percentile: 441
99th percentile: 801
Max: 55179

Chosen MAX_LEN: 441
X_train_seq: (106510, 441)
X_val_seq:   (18797, 441)
X_test_seq:  (22113, 441)


# Build the BiGRU model
This cell builds a word-level BiGRU model for binary era classification.

In [47]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Bidirectional, Dense, Dropout, Input

rnn_vocab_size = min(VOCAB_SIZE, len(tokenizer_rnn.word_index) + 1)

bigru_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(input_dim=rnn_vocab_size, output_dim=128),
    Bidirectional(GRU(32, return_sequences=True)),
    Dropout(0.7),
    Bidirectional(GRU(32)),
    Dropout(0.7),
    Dense(1, activation="sigmoid")
])

bigru_model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

bigru_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 441, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 441, 64)        │        31,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 441, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        18,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,609,985 (9.96 MB)

 Trainable params: 2,609,985 (9.96 MB)

 Non-trainable params: 0 (0.00 B)

In [48]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stopping_bigru = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

checkpoint_bigru = ModelCheckpoint(
    "best_bigru.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

history_bigru = bigru_model.fit(
    X_train_seq,
    y_train,
    validation_data=(X_val_seq, y_val),
    epochs=10,
    batch_size=128,
    callbacks=[early_stopping_bigru, checkpoint_bigru],
    verbose=1
)

Epoch 1/10
833/833 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.6540 - loss: 0.6327
Epoch 1: val_accuracy improved from None to 0.69830, saving model to best_bigru.keras

Epoch 1: finished saving model to best_bigru.keras
833/833 ━━━━━━━━━━━━━━━━━━━━ 56s 61ms/step - accuracy: 0.6925 - loss: 0.5951 - val_accuracy: 0.6983 - val_loss: 0.6254
Epoch 2/10
833/833 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.7720 - loss: 0.5037
Epoch 2: val_accuracy improved from 0.69830 to 0.80997, saving model to best_bigru.keras

Epoch 2: finished saving model to best_bigru.keras
833/833 ━━━━━━━━━━━━━━━━━━━━ 50s 60ms/step - accuracy: 0.7894 - loss: 0.4777 - val_accuracy: 0.8100 - val_loss: 0.4343
Epoch 3/10
833/833 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8209 - loss: 0.4238
Epoch 3: val_accuracy improved from 0.80997 to 0.81683, saving model to best_bigru.keras

Epoch 3: finished saving model to best_bigru.keras
833/833 ━━━━━━━━━━━━━━━━━━━━ 49s 59ms/step - accuracy: 0.8287 - loss: 0.4108 - va

# Evaluate the BiGRU model on the test set
This cell loads the best BiGRU model and reports the final test metrics on the held-out test set.

In [49]:
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

best_bigru_model = load_model("best_bigru.keras")

y_proba_bigru = best_bigru_model.predict(X_test_seq, batch_size=128, verbose=1)
y_pred_bigru = (y_proba_bigru >= 0.5).astype("int32").reshape(-1)

accuracy_bigru = accuracy_score(y_test, y_pred_bigru)
precision_bigru, recall_bigru, f1_bigru, _ = precision_recall_fscore_support(
    y_test, y_pred_bigru, average="macro"
)

print(f"Test Accuracy:   {accuracy_bigru:.4f}")
print(f"Macro Precision: {precision_bigru:.4f}")
print(f"Macro Recall:    {recall_bigru:.4f}")
print(f"Macro F1:        {f1_bigru:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_bigru,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_bigru))

173/173 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step
Test Accuracy:   0.8142
Macro Precision: 0.8118
Macro Recall:    0.7839
Macro F1:        0.7934

Classification Report:
              precision    recall  f1-score   support

   Classical       0.82      0.90      0.86     13842
      Modern       0.81      0.66      0.73      8271

    accuracy                           0.81     22113
   macro avg       0.81      0.78      0.79     22113
weighted avg       0.81      0.81      0.81     22113

Confusion Matrix:
[[12516  1326]
 [ 2782  5489]]


In [50]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/lsen"
os.makedirs(save_dir, exist_ok=True)

source_model_path = "best_bigru.keras"
target_model_path = os.path.join(save_dir, "BIGRUera.keras")

shutil.copy2(source_model_path, target_model_path)

print("Model saved to:")
print(target_model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model saved to:
/content/drive/MyDrive/lsen/BIGRUera.keras


In [63]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the saved BiGRU model if needed
bigru_infer_model = load_model("best_bigru.keras")

def predict_era_bigru(poem_text, model, tokenizer, max_len):
    cleaned_text = clean_arabic_text(poem_text)
    seq = tokenizer.texts_to_sequences([cleaned_text])
    seq_pad = pad_sequences(seq, maxlen=max_len, padding="post", truncating="post")

    prob = model.predict(seq_pad, verbose=0)[0][0]
    pred_label = "Modern" if prob >= 0.5 else "Classical"

    return {
        "original_text": poem_text,
        "cleaned_text": cleaned_text,
        "modern_probability": float(prob),
        "classical_probability": float(1 - prob),
        "predicted_label": pred_label
    }

# Put your poem here
sample_poem = """
لآخرِ أمر المـرء تُفضي أَوائِلُـه
و دهْرُكَ في الإدبار كُثر وسائلُهْ
جَرَعتُ كؤوس الهمِّ حتى ألِفْتُها
و قد ضاق ذرعا بي وأُثْقِلَ كاهِلُهْ
وأَبْدى ليَ الأصحابُ ما لسْتُ جاهلاً
بأنَّ اشْتِدادَ الكرب فَرداً تُعَامِلُـهْ
وأنْ لا وفاء غير في الأُنس والرَّخا
وإلّا فَكُلٌّ قَد توالَت مَشاغِلُـه
عَرَفتُ من الأيام تكرار غدرِها
فمَن قالَ أنَّ الغدر قد خابَ فاعِلُهْ
فما خابت الأيامُ من سوء فعلها
وقدخاب حُسْنُ الظَّن خابَ تفاؤلُه
فيا ظَهري المطعون بالخنجر الذّي
على خيري الموصول قد شَبَّ حامِلُه
أيَا ظَهري المطعون خيراً فعَلْت بي
بألّا تُريني الغدر وجهاً أُقابِلُهْ
أيا فالقَ الإصباحِ والحَبِّ والنَّوى
ويا سابقاً للعفو تُخْشَى نَوازِلُهْ
فَلَم يَبقَ إلّا أنتَ منجا ومُرْتجى
وإلّا فَكُلٌّ قد توَلَّت قوافِلُه

"""

result = predict_era_bigru(sample_poem, bigru_infer_model, tokenizer_rnn, MAX_LEN)

print("Predicted label:", result["predicted_label"])
print("Modern probability:", round(result["modern_probability"], 4))
print("Classical probability:", round(result["classical_probability"], 4))
print("\nCleaned text:")
print(result["cleaned_text"][:1000])

Predicted label: Modern
Modern probability: 0.9482
Classical probability: 0.0518

Cleaned text:
لآخر أمر المرء تفضي أوائله و دهرك في الإدبار كثر وسائله جرعت كؤوس الهم حتى ألفتها و قد ضاق ذرعا بي وأثقل كاهله وأبدى لي الأصحاب ما لست جاهلا بأن اشتداد الكرب فردا تعامله وأن لا وفاء غير في الأنس والرخا وإلا فكل قد توالت مشاغله عرفت من الأيام تكرار غدرها فمن قال أن الغدر قد خاب فاعله فما خابت الأيام من سوء فعلها وقدخاب حسن الظن خاب تفاؤله فيا ظهري المطعون بالخنجر الذي على خيري الموصول قد شب حامله أيا ظهري المطعون خيرا فعلت بي بألا تريني الغدر وجها أقابله أيا فالق الإصباح والحب والنوى ويا سابقا للعفو تخشى نوازله فلم يبق إلا أنت منجا ومرتجى وإلا فكل قد تولت قوافله


In [52]:
!pip -q install transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


# Load Arabic BERT tokenizer and model
This cell loads an Arabic BERT tokenizer and sequence classification model for binary classification.

In [53]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "aubmindlab/bert-base-arabertv02"

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print("Loaded model:", MODEL_NAME)
print("Number of labels:", bert_model.config.num_labels)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Loaded model: aubmindlab/bert-base-arabertv02
Number of labels: 2


# Prepare a smaller BERT dataset split
This cell builds DataFrames for train, validation, and test using the existing cleaned text and labels.
For a first BERT experiment, it uses a smaller subset to keep training practical in Colab.

In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Rebuild small DataFrames from the already-split arrays
train_df_bert = pd.DataFrame({"text": X_train, "label": y_train})
val_df_bert   = pd.DataFrame({"text": X_val, "label": y_val})
test_df_bert  = pd.DataFrame({"text": X_test, "label": y_test})

# Smaller subset for a practical first BERT run in Colab
train_df_bert_small, _ = train_test_split(
    train_df_bert,
    train_size=20000,
    random_state=SEED,
    stratify=train_df_bert["label"]
)

val_df_bert_small, _ = train_test_split(
    val_df_bert,
    train_size=4000,
    random_state=SEED,
    stratify=val_df_bert["label"]
)

test_df_bert_small, _ = train_test_split(
    test_df_bert,
    train_size=4000,
    random_state=SEED,
    stratify=test_df_bert["label"]
)

print("Train subset shape:", train_df_bert_small.shape)
print("Validation subset shape:", val_df_bert_small.shape)
print("Test subset shape:", test_df_bert_small.shape)

print("\nTrain label distribution:")
print(train_df_bert_small["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(val_df_bert_small["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(test_df_bert_small["label"].value_counts().sort_index())

display(train_df_bert_small.head(3))

Train subset shape: (20000, 2)
Validation subset shape: (4000, 2)
Test subset shape: (4000, 2)

Train label distribution:
label
0    12520
1     7480
Name: count, dtype: int64

Validation label distribution:
label
0    2504
1    1496
Name: count, dtype: int64

Test label distribution:
label
0    2504
1    1496
Name: count, dtype: int64


,text,label
106457,هي الأواصر أدناها الدم الجاري فلا محالة من حب ...,1
33126,يا بني هاجر وتب لكم ما هذه الكبرياء والعظمه نا...,0
72511,قد جار والله على جاره والله قد أوصاه بالجار حت...,0


# Tokenize text for BERT
This cell tokenizes the text using the Arabic BERT tokenizer and converts the subsets into Hugging Face datasets.

In [56]:
import numpy as np
import evaluate

from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

    return {
        "accuracy": acc,
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1
    }

training_args = TrainingArguments(
    output_dir="./bert_era_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    fp16=True
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_ds_bert,
    eval_dataset=val_ds_bert,
    compute_metrics=compute_metrics
)

print("Trainer is ready.")

Trainer is ready.


# Train the BERT model
This cell fine-tunes the Arabic BERT model on the binary era classification task.

In [57]:
train_result = trainer.train()
train_result

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.527659,0.465436,0.795500,0.814391,0.745979,0.760022
2,0.401383,0.452572,0.818500,0.812393,0.793411,0.800615
3,0.319556,0.664147,0.818250,0.808982,0.797920,0.802578


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=7500, training_loss=0.416199658203125, metrics={'train_runtime': 386.5344, 'train_samples_per_second': 155.226, 'train_steps_per_second': 19.403, 'total_flos': 7893331660800000.0, 'train_loss': 0.416199658203125, 'epoch': 3.0})

# Evaluate BERT on the test set
This cell evaluates the fine-tuned Arabic BERT model on the held-out test subset and reports the final metrics.

In [58]:
test_metrics = trainer.evaluate(test_ds_bert)
test_metrics

{'eval_loss': 0.7110443115234375,
 'eval_accuracy': 0.81125,
 'eval_macro_precision': 0.8028353395029196,
 'eval_macro_recall': 0.7876200218687532,
 'eval_macro_f1': 0.793623358118348,
 'eval_runtime': 8.0136,
 'eval_samples_per_second': 499.154,
 'eval_steps_per_second': 62.394,
 'epoch': 3.0}

# Confusion matrix for BERT
This cell gets BERT predictions on the test subset, prints the classification report, and shows the confusion matrix.

In [59]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions
pred_output = trainer.predict(test_ds_bert)
y_true_bert = pred_output.label_ids
y_pred_bert = np.argmax(pred_output.predictions, axis=-1)

print("Classification Report:")
print(classification_report(
    y_true_bert,
    y_pred_bert,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true_bert, y_pred_bert))

Classification Report:
              precision    recall  f1-score   support

   Classical       0.83      0.88      0.85      2504
      Modern       0.78      0.69      0.73      1496

    accuracy                           0.81      4000
   macro avg       0.80      0.79      0.79      4000
weighted avg       0.81      0.81      0.81      4000

Confusion Matrix:
[[2207  297]
 [ 458 1038]]


In [60]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/lsen/BERTera"
os.makedirs(save_dir, exist_ok=True)

trainer.save_model(save_dir)
bert_tokenizer.save_pretrained(save_dir)

print("BERT model and tokenizer saved to:")
print(save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model and tokenizer saved to:
/content/drive/MyDrive/lsen/BERTera


# Prepare full BERT datasets
This cell rebuilds the full train, validation, and test datasets for BERT using all available cleaned texts and labels.

In [65]:
from datasets import Dataset
import pandas as pd

# Full DataFrames from the existing splits
train_df_bert_full = pd.DataFrame({"text": X_train, "label": y_train})
val_df_bert_full   = pd.DataFrame({"text": X_val, "label": y_val})
test_df_bert_full  = pd.DataFrame({"text": X_test, "label": y_test})

print("Train full shape:", train_df_bert_full.shape)
print("Validation full shape:", val_df_bert_full.shape)
print("Test full shape:", test_df_bert_full.shape)

train_ds_bert_full = Dataset.from_pandas(train_df_bert_full.reset_index(drop=True))
val_ds_bert_full   = Dataset.from_pandas(val_df_bert_full.reset_index(drop=True))
test_ds_bert_full  = Dataset.from_pandas(test_df_bert_full.reset_index(drop=True))

print(train_ds_bert_full)
print(val_ds_bert_full)
print(test_ds_bert_full)

Train full shape: (106510, 2)
Validation full shape: (18797, 2)
Test full shape: (22113, 2)
Dataset({
    features: ['text', 'label'],
    num_rows: 106510
})
Dataset({
    features: ['text', 'label'],
    num_rows: 18797
})
Dataset({
    features: ['text', 'label'],
    num_rows: 22113
})


# Tokenize full BERT datasets
This cell tokenizes the full train, validation, and test datasets using the Arabic BERT tokenizer and prepares them for training.

# Set up full BERT training
This cell reloads a fresh Arabic BERT model, defines the evaluation metrics, and prepares the Trainer for full-data fine-tuning.

In [67]:
import numpy as np
import evaluate

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Reload a fresh model for full-data training
MODEL_NAME = "aubmindlab/bert-base-arabertv02"

bert_model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    prec = precision_metric.compute(predictions=preds, references=labels, average="macro")["precision"]
    rec = recall_metric.compute(predictions=preds, references=labels, average="macro")["recall"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]

    return {
        "accuracy": acc,
        "macro_precision": prec,
        "macro_recall": rec,
        "macro_f1": f1
    }

training_args_full = TrainingArguments(
    output_dir="./bert_era_full_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    fp16=True
)

trainer_full = Trainer(
    model=bert_model_full,
    args=training_args_full,
    train_dataset=train_ds_bert_full,
    eval_dataset=val_ds_bert_full,
    compute_metrics=compute_metrics
)

print("Full-data BERT trainer is ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Full-data BERT trainer is ready.


# Train BERT on the full dataset
This cell fine-tunes Arabic BERT using the full train and validation sets.

In [68]:
train_result_full = trainer_full.train()
train_result_full

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.506290,0.449170,0.819120,0.808422,0.801986,0.804886
2,0.413352,0.406472,0.828962,0.816841,0.819697,0.818197
3,0.358573,0.481096,0.837155,0.825940,0.826698,0.826315


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=39942, training_loss=0.42607140461090454, metrics={'train_runtime': 2035.5108, 'train_samples_per_second': 156.978, 'train_steps_per_second': 19.623, 'total_flos': 4.20359377595904e+16, 'train_loss': 0.42607140461090454, 'epoch': 3.0})

# Evaluate full-data BERT on the full test set
This cell evaluates the fully fine-tuned Arabic BERT model on the full held-out test set and reports the final metrics.

In [69]:
test_metrics_full = trainer_full.evaluate(test_ds_bert_full)
test_metrics_full

{'eval_loss': 0.499736487865448,
 'eval_accuracy': 0.8324514991181657,
 'eval_macro_precision': 0.8207600765539229,
 'eval_macro_recall': 0.8223980916920464,
 'eval_macro_f1': 0.8215565111412484,
 'eval_runtime': 43.6958,
 'eval_samples_per_second': 506.067,
 'eval_steps_per_second': 63.278,
 'epoch': 3.0}

# Confusion matrix for full-data BERT
This cell gets predictions from the fully fine-tuned BERT model on the full test set, prints the classification report, and shows the confusion matrix.

In [70]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_output_full = trainer_full.predict(test_ds_bert_full)
y_true_bert_full = pred_output_full.label_ids
y_pred_bert_full = np.argmax(pred_output_full.predictions, axis=-1)

print("Classification Report:")
print(classification_report(
    y_true_bert_full,
    y_pred_bert_full,
    target_names=["Classical", "Modern"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_true_bert_full, y_pred_bert_full))

Classification Report:
              precision    recall  f1-score   support

   Classical       0.87      0.86      0.87     13842
      Modern       0.77      0.78      0.78      8271

    accuracy                           0.83     22113
   macro avg       0.82      0.82      0.82     22113
weighted avg       0.83      0.83      0.83     22113

Confusion Matrix:
[[11936  1906]
 [ 1799  6472]]


In [71]:
from google.colab import drive
import os

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/lsen/BERTerav2"
os.makedirs(save_dir, exist_ok=True)

trainer_full.save_model(save_dir)
bert_tokenizer.save_pretrained(save_dir)

print("Full-data BERT model and tokenizer saved to:")
print(save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Full-data BERT model and tokenizer saved to:
/content/drive/MyDrive/lsen/BERTerav2


In [72]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

bert_v2_dir = "/content/drive/MyDrive/lsen/BERTerav2"

bert_v2_tokenizer = AutoTokenizer.from_pretrained(bert_v2_dir)
bert_v2_model = AutoModelForSequenceClassification.from_pretrained(bert_v2_dir)
bert_v2_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_v2_model.to(device)

def predict_era_bert_v2(poem_text, model, tokenizer, max_len=256):
    cleaned_text = clean_arabic_text(poem_text)

    encoded = tokenizer(
        cleaned_text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    label_names = ["Classical", "Modern"]

    return {
        "original_text": poem_text,
        "cleaned_text": cleaned_text,
        "predicted_label": label_names[pred_idx],
        "classical_probability": float(probs[0]),
        "modern_probability": float(probs[1]),
    }

# Put your poem here
sample_poem = """
لآخرِ أمر المـرء تُفضي أَوائِلُـه
و دهْرُكَ في الإدبار كُثر وسائلُهْ
جَرَعتُ كؤوس الهمِّ حتى ألِفْتُها
و قد ضاق ذرعا بي وأُثْقِلَ كاهِلُهْ
وأَبْدى ليَ الأصحابُ ما لسْتُ جاهلاً
بأنَّ اشْتِدادَ الكرب فَرداً تُعَامِلُـهْ
وأنْ لا وفاء غير في الأُنس والرَّخا
وإلّا فَكُلٌّ قَد توالَت مَشاغِلُـه
عَرَفتُ من الأيام تكرار غدرِها
فمَن قالَ أنَّ الغدر قد خابَ فاعِلُهْ
فما خابت الأيامُ من سوء فعلها
وقدخاب حُسْنُ الظَّن خابَ تفاؤلُه
فيا ظَهري المطعون بالخنجر الذّي
على خيري الموصول قد شَبَّ حامِلُه
أيَا ظَهري المطعون خيراً فعَلْت بي
بألّا تُريني الغدر وجهاً أُقابِلُهْ
أيا فالقَ الإصباحِ والحَبِّ والنَّوى
ويا سابقاً للعفو تُخْشَى نَوازِلُهْ
فَلَم يَبقَ إلّا أنتَ منجا ومُرْتجى
وإلّا فَكُلٌّ قد توَلَّت قوافِلُه"""

result = predict_era_bert_v2(sample_poem, bert_v2_model, bert_v2_tokenizer, max_len=256)

print("Predicted label:", result["predicted_label"])
print("Classical probability:", round(result["classical_probability"], 4))
print("Modern probability:", round(result["modern_probability"], 4))
print("\nCleaned text:")
print(result["cleaned_text"][:1000])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicted label: Modern
Classical probability: 0.0084
Modern probability: 0.9916

Cleaned text:
لآخر أمر المرء تفضي أوائله و دهرك في الإدبار كثر وسائله جرعت كؤوس الهم حتى ألفتها و قد ضاق ذرعا بي وأثقل كاهله وأبدى لي الأصحاب ما لست جاهلا بأن اشتداد الكرب فردا تعامله وأن لا وفاء غير في الأنس والرخا وإلا فكل قد توالت مشاغله عرفت من الأيام تكرار غدرها فمن قال أن الغدر قد خاب فاعله فما خابت الأيام من سوء فعلها وقدخاب حسن الظن خاب تفاؤله فيا ظهري المطعون بالخنجر الذي على خيري الموصول قد شب حامله أيا ظهري المطعون خيرا فعلت بي بألا تريني الغدر وجها أقابله أيا فالق الإصباح والحب والنوى ويا سابقا للعفو تخشى نوازله فلم يبق إلا أنت منجا ومرتجى وإلا فكل قد تولت قوافله


# Using `BERTerav2` Project

This section explains how to use the final trained model in project.

The model takes a poem as input and predicts one of two classes:

- **Classical**
- **Modern**

---

## Workflow

To use the model correctly, follow these steps in order:

1. **Load the saved model and tokenizer**
2. **Clean the input poem**
3. **Tokenize the cleaned text**
4. **Run inference**
5. **Read the predicted label and probabilities**




دوال جاهزه للاستخدام لربطه بالمميزات



In [ ]:
#Load model and tokenizer
MODEL_DIR = "تستت"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()






# Preprocessing function
arabic_diacritics = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')
kashida_char = '\u0640'

def clean_arabic_text(text):
    if text is None:
        return ""

    text = html.unescape(str(text))

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove diacritics
    text = re.sub(arabic_diacritics, "", text)

    # Remove kashida
    text = text.replace(kashida_char, "")

    # Remove digits
    text = re.sub(r"[0-9٠-٩]", " ", text)

    # Keep Arabic letters and spaces only
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text






    #Main prediction function
    def predict_poem_era(poem_text, model, tokenizer, device, max_len=256):
    cleaned_text = clean_arabic_text(poem_text)

    encoded = tokenizer(
        cleaned_text,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

    encoded = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    label_names = ["Classical", "Modern"]
    pred_idx = int(np.argmax(probs))

    return {
        "original_text": poem_text,
        "cleaned_text": cleaned_text,
        "predicted_label": label_names[pred_idx],
        "classical_probability": float(probs[0]),
        "modern_probability": float(probs[1]),
    }




    #Example usage
    sample_poem = """
قفا نبك من ذكرى حبيب ومنزل
بسقط اللوى بين الدخول فحومل
"""

result = predict_poem_era(sample_poem, model, tokenizer, device)

print("Predicted label:", result["predicted_label"])
print("Classical probability:", round(result["classical_probability"], 4))
print("Modern probability:", round(result["modern_probability"], 4))
print("Cleaned text:", result["cleaned_text"])